# 回帰(Regression)と分類(Classification)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyorin-phys/MLIntro2026/blob/main/3/3.ipynb)

<img src="linear-vs-logistic.png" width="75%">


image from [ML for Beginners](https://github.com/microsoft/ML-For-Beginners/tree/main/2-Regression/4-Logistic)

* 目的関数の違い：　
スケール変数とカテゴリー変数

* 線形回帰：
1変数、多変数

* ロジスティック回帰(**回帰**という名前だが実際は**分類**が目的)：

    * 2値（0か1か、陽性か陰性か）： sigmoid関数で陽性である確率を求める

    * 多クラス（3種類以上のどれか）: softmax関数で各クラスの確率を求め、最も高いクラスを選ぶ

    この回では、まず2値分類で使い方と評価の仕方を学び、最後に多クラス分類に広げる。

数値データなら深層学習でなくてもできる

アヤメデータを使う

* 特徴量：がく片、花びらの幅と長さ　４変数
* 目的変数：花の種類

<img src="https://miro.medium.com/max/1000/1*Hh53mOF4Xy4eORjLilKOwA.png" width="50%">

## seaborn ライブラリ　
統計用グラフが充実
* [seaborn](https://seaborn.pydata.org/index.html)

[グラフ一覧](https://seaborn.pydata.org/examples/index.html)

[matplotlib & seaborn](https://www.amazon.co.jp/dp/4798055433) これだけで本になるくらいなので使いこなすのは大変だが、そのまま出版に使えるような品質の高いグラフが作れる。関数名で検索すれば、用例を解説したページが多数ある。

In [ ]:
# アヤメデータ(前回はCSVファイルを使ったが、データセットがライブラリに用意されている)
import seaborn as sns # なぜSNSと略すのかはアメリカのテレビ番組が由来なので説明されてもわからない
# from sklearn.datasets import load_iris # sklearn からデータを読むこともできる
print(sns.get_dataset_names()) # seabornに含まれるサンプルデータ
iris = sns.load_dataset('iris')
# このデータはpandasのDataFrame形式になっている
iris.head() 

In [ ]:
# petal_length, petal_widthで散布図、hue で種ごとに色分け
sns.relplot(data=iris, x='petal_length',y='petal_width', hue='species')

各特徴量のペア毎の散布図をまとめて描いたり、相関係数を視覚的に確認できる。

In [ ]:
# 散布図行列　
sns.pairplot(iris, hue='species', corner=True) # 対角線をまたいで対称なので下半分だけ表示

### 3次元の散布図
[plotly](https://plotly.com/python/) を使うと、マウスで回転・拡大できる3次元のグラフが描ける。3つの特徴量を選んで、種の分かれ方を立体的に見てみよう。

setosa は他の2種から離れているが、versicolor と virginica は接していて一部が重なっている。また versicolor は setosa と virginica の間にある。

In [ ]:
# 3次元散布図（マウスで回転、ホイールで拡大できる）
import plotly.express as px
fig = px.scatter_3d(iris, x='petal_length', y='petal_width', z='sepal_length', color='species', width=700, height=500)
fig.update_traces(marker_size=4)
fig.show()

In [ ]:
# 相関行列
corr = iris.corr(numeric_only=True) #カテゴリー変数speciesを除く
corr

In [ ]:
# 相関行列のヒートマップ
sns.heatmap(corr, cmap='coolwarm',annot=True)

## [scikit-learn](https://scikit-learn.org/stable/index.html) (sklearn)

データ分析に有用なライブラリ
* LinearRegression: 線形回帰
* LogisticRegression: ロジスティック回帰
* train_test_split: 訓練用と検証用にデータを分割する

supervised learning(教師あり学習): 特徴量と目的変数のセットが必要

<img src='split.png' width='50%'>

訓練データで回帰式を求め、検証データでどれくらい再現できているかを検証する

* root_mean_squared_error, r2_score: 
結果の良し悪しを判断 (二乗平均平方根誤差、決定係数（単回帰では相関係数の2乗）)

* 線形回帰、ロジスティック回帰、その他の方法が全て同じ構文で統一されている。
```
model = モデル名() # モデルを指定
model.fit() # 学習
model.predict() # 予測
```


In [ ]:
# 準備
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error

# アヤメデータセットを読み込み
iris = sns.load_dataset('iris') #DataFrame形式で読み込み　
iris.head()

In [ ]:
# ファイルに書き出しできる
#iris.to_csv('iris.csv', index=False) # 1列目にインデックスを追加しない
#iris.to_excel('iris.xlsx', index=False)
#df = pd.read_csv('iris.csv') # 読み込み
#df = pd.read_excel('iris.xlsx')
#df.head()

In [ ]:
# setosa だけ抽出したいとき
setosa = iris[iris['species']=='setosa']
setosa.tail()

In [ ]:
# 花びら(petal)の長さと幅のデータを抽出
# 単回帰
X = iris[['petal_length']] # ベクトルではなく、行列形式
y = iris['petal_width']    # ベクトル

# 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print(X_train.size, X_test.size, len(X)*0.3) # Xのサイズにtest_sizeを掛けた数が検証用データの個数

# 線形回帰モデルを作成
model = LinearRegression()

# モデルを訓練
model.fit(X_train, y_train)

# テストデータで予測
y_pred = model.predict(X_test)

print(y_pred.round(2))
print(y_test.values)

print('傾き:',model.coef_[0], 'y切片', model.intercept_)

In [ ]:
# モデルの評価
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Root Mean Squared Error: {rmse}')
print(f'R^2 Score: {r2}')

# 回帰直線をプロット
#plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, color='blue', label='Actual data')
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Regression line')
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.title('Linear Regression: Petal Length vs Petal Width')
plt.legend()
plt.show()

## ロジスティック回帰（2値分類）

目的変数がカテゴリー変数のときに使う（**回帰**という名前だが実際は**分類**）。
まず目的変数が**2値**（陽性=1 / 陰性=0）の場合を扱う。

* `LogisticRegression` ロジスティック回帰
* `train_test_split` データを訓練用(train)、検証用(test)に分割する
* `accuracy_score` 正解率
* 混同行列、感度・特異度、ROC曲線とAUC（乳がんの例で説明する）

### 2値の例1：setosa かどうか
アヤメの種がsetosaであるかどうかをロジスティック回帰で予測する

In [ ]:
# 準備
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression # ロジスティック回帰
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score # 2値分類なので平均二乗誤差を使わない

# アヤメデータセットを読み込み
iris = load_iris()
X = iris.data
y = iris.target
y
# y=0: setosa, y=1: versicolor, y=2: virginica
# それぞれの種が50個ずつ

In [ ]:
# setosa なら1、その他は0 
y_bin = (y==0).astype(int) # bool型numpy配列をまとめてint型に変換
y_bin

In [ ]:
X.size, y_bin.size # Xは4変数x150件, yは150件

In [ ]:
# setosa かどうかを2値分類する
# 訓練データとテストデータに分割（順序をシャッフルし、乱数を固定するためにrandom_stateを指定）
X_train, X_test, y_train, y_test = train_test_split(X, y_bin, test_size=0.2, random_state=0)

# ロジスティック回帰モデルを作成
model = LogisticRegression()

# モデルを訓練
model.fit(X_train, y_train)

# テストデータで予測
y_pred = model.predict(X_test)

print('予測', y_pred)
print('正解', y_test)

print('正解率', accuracy_score(y_test, y_pred))

In [ ]:
150*0.2, len(y_pred) # 150件のうち20%を検証用に使った

`test_size`, `random_state`の値を変えると結果が変わるかどうか試してみよ。

test_sizeを大きくすると学習データが減るので、正解率は下がるはずだが、setosaは孤立しているので、ほとんど変わらない。

random_stateは乱数の出方を変えるだけなので、データの選択によって偶然変わることがあるが、本質的には変わらないはず。

setosa以外の種について同様にやってみよ。

In [ ]:
# 3種それぞれについて「その種かどうか」の2値分類を行い、正解率を比べる
for k, name in enumerate(iris.target_names):
    y_k = (y==k).astype(int) # その種なら1、それ以外は0
    X_train, X_test, y_train, y_test = train_test_split(X, y_k, test_size=0.2, random_state=0)
    model = LogisticRegression(max_iter=1000) # 計算が収束するよう繰り返し回数を増やす
    model.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, model.predict(X_train))
    acc_test = accuracy_score(y_test, model.predict(X_test))
    print(f'{name:10s} 訓練データの正解率: {acc_train:.3f}  検証データの正解率: {acc_test:.3f}')

* **setosa**：他の2種から離れているので、完全に分けられる。
* **virginica**：versicolor と一部重なるので、訓練データでも100%にはならない。検証データで1.0になるのは、たまたま紛らわしいデータが検証用の30件に入らなかったためで、`random_state` を変えると下がる。
* **versicolor**：花びらの大きさが setosa と virginica の**中間**にあるので、「versicolor か、それ以外か」は1本の直線（4変数なら平面）では分けられない（両側に「それ以外」がある）。ロジスティック回帰は直線で分ける方法なので、正解率は低い。（上の3次元散布図で確かめてみよ）

このように、正解率はデータの分かれ方やデータの選び方で大きく変わる。次に、より現実的な2値分類として乳がんの診断を扱い、正解率以外の評価指標を学ぶ。

### 2値の例2：乳がんの診断
breast cancer データ（569人、細胞核の画像から求めた30個の特徴量）を使って、悪性かどうかをロジスティック回帰で予測する。

元のデータは悪性=0、良性=1なので、**悪性を1（陽性）**に変換してから使う。

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

cancer = load_breast_cancer()
# target: 0=悪性(malignant), 1=良性(benign)。悪性が0なので注意
print(cancer.DESCR)
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
df.head()

### 混同行列(confusion matrix)と評価指標

予測と実際の組み合わせを数えた表を混同行列という。

| 実際　\　予測    | 陽性 | 陰性 |
|----|----|----|
|陽性 | TP（真陽性）   | FN（偽陰性、見逃し）   |
|陰性　| FP（偽陽性） | TN（真陰性） |

* 正解率：全体の中で正しく分類できた割合

$$\mathrm{accuracy = \frac{TP+TN}{TP+TN+FP+FN}}$$

* 感度（再現率 recall）：実際に陽性の人のうち、正しく陽性と予測できた割合

$$\mathrm{sensitivity = recall = \frac{TP}{TP+FN}}$$

* 特異度：実際に陰性の人のうち、正しく陰性と予測できた割合

$$\mathrm{specificity = \frac{TN}{TN+FP}}$$

* 適合率（精度 precision、陽性的中率）：陽性と予測された中で実際に陽性だった割合

$$\mathrm{precision = \frac{TP}{TP+FP}}$$

* F1スコア： 適合率と再現率の調和平均（バランスを取る。どちらかが極端に高い・低い場合に有用）

$$\mathrm{F1score = \frac{2\times precision \times recall}{precision + recall}}$$

医療の検査では感度と特異度がよく使われる。機械学習の分野では precision と recall がよく使われ、`classification_report` で表示される。

※ scikit-learn の `confusion_matrix` も行が実際、列が予測だが、並び順はラベルの番号順（0, 1, ...）になる。陽性=1とすると、左上がTN、右下がTPとなり、上の表とは並びが逆になるので注意。

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

cancer = load_breast_cancer()
X = cancer.data
y = (cancer.target == 0).astype(int) # 悪性を1(陽性)、良性を0(陰性)に変換

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

model = LogisticRegression(max_iter=10000) # 特徴量の大きさがばらばらなので、計算が収束するまで繰り返し回数を増やす
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print(cm)
tn, fp, fn, tp = cm.ravel() # 並び順は [[TN, FP], [FN, TP]]
print('感度:', tp / (tp + fn))
print('特異度:', tn / (tn + fp))
print(classification_report(y_test, y_pred, target_names=['benign', 'malignant']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['benign', 'malignant'])
plt.show()

がんの診断では、悪性を見逃す（FN）ほうが、良性を悪性と判定する（FP）よりも問題が大きい。

`model.predict()` は悪性である確率が0.5以上なら悪性と予測している。この境目（閾値。医学の検査ではカットオフ値という）を下げると、見逃しは減るが、良性を悪性と判定する数は増える。つまり感度は上がり、特異度は下がる。*

In [ ]:
# predict_proba で各クラスの確率を求める。[:, 1] は悪性(1)である確率
prob = model.predict_proba(X_test)[:, 1]

for threshold in [0.5, 0.3, 0.1]:
    y_pred2 = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred2).ravel()
    print(f'閾値 {threshold}: 見逃し(FN) {fn}件, 良性を悪性と判定(FP) {fp}件, 感度 {tp/(tp+fn):.3f}, 特異度 {tn/(tn+fp):.3f}')

### ROC曲線とAUC*
閾値を1から0まで少しずつ下げながら、横軸に偽陽性率（1 − 特異度）、縦軸に感度をとって結んだ曲線をROC曲線という。検査や診断法の性能評価によく使われる。

* 左上の角（偽陽性率0、感度1）に近いほど良い。
* 曲線の下の面積をAUC（Area Under the Curve）といい、1に近いほど良い。でたらめに判定すると対角線になり、AUCは0.5になる。
* AUCは閾値を決める前の、モデルそのものの性能を表す。

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_test, prob) # 偽陽性率、感度、そのときの閾値
auc = roc_auc_score(y_test, prob)
print('AUC:', auc)

# 左: 全体、右: 左上の角を拡大
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax in axes:
    ax.plot(fpr, tpr, label=f'ROC (AUC = {auc:.3f})')
    ax.plot([0, 1], [0, 1], color='gray', linestyle='--', label='random') # でたらめに判定した場合
    ax.set_xlabel('False Positive Rate (1 - specificity)')
    ax.set_ylabel('True Positive Rate (sensitivity)')
axes[0].legend()
axes[1].set_xlim(0, 0.3)
axes[1].set_ylim(0.8, 1.02)

# 拡大図に、上で試した閾値の位置を点で示す（点の上の数字が閾値）
for threshold in [0.5, 0.3, 0.1]:
    tn, fp, fn, tp = confusion_matrix(y_test, (prob >= threshold).astype(int)).ravel()
    px, py = fp / (fp + tn), tp / (tp + fn) # 偽陽性率、感度
    axes[1].scatter(px, py, zorder=3)
    axes[1].annotate(f'{threshold}', (px, py), textcoords='offset points', xytext=(0, 8), ha='center')

plt.tight_layout()
plt.show()

#### 自分のデータでROC曲線を描くには
`RocCurveDisplay.from_predictions(正解, 数値)` の1行で描ける。

* 正解：陽性=1、陰性=0 の配列
* 数値：陽性らしいほど大きくなる値。機械学習の予測確率でなくても、**検査値そのもの**でもよい

例えば、ある検査値で病気を見分けられるかを調べるときは、患者ごとの「病気の有無」と「検査値」を表にしておけば、この1行でROC曲線とAUCが得られる。下の例では、30個の特徴量を使ったロジスティック回帰と、1つの検査値だけの場合のAUCを比べられる。

In [ ]:
from sklearn.metrics import RocCurveDisplay

# 機械学習の予測確率を使う場合
RocCurveDisplay.from_predictions(y_test, prob, name='logistic regression')
plt.show()

# 検査値をそのまま使う場合
# 例: 細胞核の半径の平均値(mean radius)を1つの検査値とみなす（値が大きいほど悪性らしい）
print(cancer.feature_names[0])
RocCurveDisplay.from_predictions(y, X[:, 0], name='mean radius')
plt.show()

## 多クラス分類
アヤメが3種（setosa, versicolor, virginica）のどれに属するかを予測する。

ロジスティック回帰は各クラスの確率を求め（softmax関数、確率の合計は1）、確率が最も高いクラスを予測とする。
使い方は2値分類と同じで、`LogisticRegression()` に 0, 1, 2 の整数のラベルをそのまま渡せばよい。

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Irisデータセットをロード
iris = load_iris()
X = iris.data
y = iris.target # 0: setosa, 1: versicolor, 2: virginica

# データをトレーニングセットとテストセットに分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)

model = LogisticRegression(max_iter=1000) # 2値分類と同じ（計算が収束するよう繰り返し回数を増やす）
model.fit(X_train, y_train)

# 各クラスの確率（1行が1件、列がクラス0,1,2。各行の合計は1）
print(model.predict_proba(X_test)[:3].round(3))

# 予測は確率が最も高いクラス
y_pred = model.predict(X_test)
print('予測', y_pred)
print('正解', y_test)
print('正解率', accuracy_score(y_test, y_pred))

### 多クラスの混同行列と評価指標
クラスが3つなので混同行列は3×3になる。2値のときと同じく行が実際、列が予測で、**対角線上が正解**、それ以外が間違いである。

陽性・陰性が1つに決まらないので、**クラスごとに「そのクラス 対 それ以外」の2値分類とみなす**と、2値と同じ指標が求められる。例えば versicolor（1）については

* TP：混同行列の対角 [1, 1]
* FN：1行目の対角以外（実際は versicolor なのに他と予測）
* FP：1列目の対角以外（実際は他なのに versicolor と予測）
* TN：残り全部

setosa は他の2種から離れているので全て正解するが、versicolor と virginica は互いに間違えやすい（散布図行列でも重なっていた）。

`classification_report` の各行は、このようにしてクラスごとに求めた precision, recall（感度）, F1 である。
`macro avg` はクラスごとの値の単純平均、`weighted avg` は各クラスの件数（support）で重みを付けた平均。

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print('混同行列')
print(cm)

# クラスごとの感度・特異度（そのクラスを陽性、それ以外を陰性とみなす）
for k in range(3):
    tp = cm[k, k]
    fn = cm[k, :].sum() - tp
    fp = cm[:, k].sum() - tp
    tn = cm.sum() - tp - fn - fp
    print(f'{iris.target_names[k]:10s} 感度: {tp/(tp+fn):.3f}  特異度: {tn/(tn+fp):.3f}')

print(classification_report(y_test, y_pred, target_names=iris.target_names))

# 混同行列の可視化
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=iris.target_names)
plt.show()

### 2値分類と多クラス分類の違い（まとめ）

| | 2値分類（乳がん） | 多クラス分類（アヤメ, K=3） |
|---|---|---|
| ラベル | 0 / 1 | 0, 1, ..., K-1 の整数 |
| モデル | `LogisticRegression()` | 同じ |
| 確率 `predict_proba` | 2列。陽性の確率は `[:, 1]` | K列。各行の合計は1 |
| 予測 `predict` | 陽性の確率が0.5以上なら1（閾値は変えられる） | 確率が最大のクラス |
| 混同行列 | 2×2（TN, FP, FN, TP） | K×K（対角が正解） |
| 感度・特異度 | 陽性クラスについて1つ | クラスごとに「そのクラス 対 それ以外」で求める |
| ROC / AUC | 1本 | クラスごとに1本。平均は `roc_auc_score(y_test, model.predict_proba(X_test), multi_class='ovr')` |

※ `sklearn.multiclass.OneVsRestClassifier` を使うと、「そのクラスかどうか」の2値分類をクラスの数だけ行い、確率が最も高いクラスを選ぶ方式になる。結果はほぼ同じになることが多い。

# その他のデータセット
sklearn.datasets にはいくつかのサンプルデータが用意されている。上で使った breast cancer の他に、回帰用のデータもある。

* [diabetes](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)　糖尿病患者　回帰用
* [breast cancer](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) 乳癌診断結果　分類用

In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

diabetes = load_diabetes()
# 特徴量（年齢、BMI、血圧、血液検査値など10項目）は平均0になるようにスケーリング済みで、元の単位の値ではない
# target: 1年後の病気の進行度を表す数値
print(diabetes.DESCR)

df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target
df.head()

# 統計用ライブラリ
* [statsmodels](https://www.statsmodels.org/stable/index.html) Rと似た形式にまとめられている。　
* [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) `linregress`という線形回帰用の関数があるが、単回帰しか扱えない。
(`sklearn.linear_model.LinearRegression`は重回帰と単回帰を同じ方法で行う。)
統計検定用の関数が各種用意されている。

これらのライブラリは統計学的解析に使われる。情報が多過ぎるのでここでは紹介するにとどめる。